## Job 3: Goodvibes model

In [2]:
load proc/atomic_model.mat Atoms SpaceGroup Basis

ENT = proc.script.ElasticNetworkTools.initialize(Atoms,Basis,SpaceGroup);
[g,Tg] = ENT.find_atom_groups();
ENT.groupAttributes = {}; % don't split into subgroups

[T,G,index] = ENT.externalModel();
ENM = ENT.exportModel(T,G);
M = ENT.M;

ResidueGroups = Tg(g(index{1}),:);

Expanding ASU contact model to the unit cell
  The ASU model has 330 edges
  After symmetry expansion there are 1320 edges     (306 internal + 1014 external)
  Of the 306 internal edges, 153 are redundant
  Deleting the redundant entries.
  Now there are 1167 edges.
Done!



## Export

In [2]:
save proc/goodvibes_model.mat ENM M ResidueGroups

## Write visualization script for chimerax

In [3]:
%%file write_springs_cmm.m

function write_springs_cmm(outDir,fileName,r1,r2,r)

[~,name]=fileparts(fileName);
name = strrep(name,'_',' ');

chimera_cmm = {'<marker_sets>',sprintf('<marker_set name="%s">',name)};

for j=1:size(r1,1)
    id1 = (j-1)*2 + 1;
    id2 = id1 + 1;
    marker_fmt = '<marker id="%d" x="%g" y="%g" z="%g" radius="%g"/>';
    link_fmt = '<link id1="%d" id2="%d" radius="%g"/>';
    chimera_cmm = [chimera_cmm,...
        {sprintf(marker_fmt,id1,r1(j,1),r1(j,2),r1(j,3),r),...
         sprintf(marker_fmt,id2,r2(j,1),r2(j,2),r2(j,3),r),...
         sprintf(link_fmt,id1,id2,r)}];
end
    chimera_cmm = [chimera_cmm,{'</marker_set>','</marker_sets>'}];

fid = fopen(fullfile(outDir,fileName),'w');
    fprintf(fid,'%s\n',chimera_cmm{:});
    fclose(fid);

end

Created file '/nfs/chess/user/spm82/mdx/mac1_20251105_goodvibes/write_springs_cmm.m'.


In [4]:
%% create bond objects representing an elastic network model

outDir = 'export';

% all
[r1,r2] = ENM.nodes();

% in chimera, set the color using
% marker change #1 color green

write_springs_cmm(outDir,'unitcell_enm_all.cmm',r1,r2,.2);

% interface
T = ENM.get_edge_info();
ifmax = max(T.interface);
for j=1:ifmax
    fn = sprintf('unitcell_enm_interface_%d.cmm',j);
    isIncl = T.interface==j;
    write_springs_cmm(outDir,fn,r1(isIncl,:),r2(isIncl,:),.2);
end

% just the ASU
isASU = ENM.Edges.o1==1;
fn = sprintf('asu_enm_all.cmm',j);
write_springs_cmm(outDir,fn,r1(isASU,:),r2(isASU,:),.2);

ifmax = max(T.interface);
for j=1:ifmax
    fn = sprintf('asu_enm_interface_%d.cmm',j);
    isIncl = T.interface==j & isASU;
    write_springs_cmm(outDir,fn,r1(isIncl,:),r2(isIncl,:),.2);
end